# 23 Build POI Weights

Build POI-level allocation weights from the cleaned OTM + Wikipedia dataset.

This notebook implements the current working methodology:
- direct Wikipedia-based signal when available
- conservative group-based fallback when Wikipedia is missing
- stricter fallback hierarchy for religious POIs
- normalized POI weight for later crowd allocation

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


In [2]:
INPUT_PATH = "../data/processed/otm_pois_model_ready.csv"
OUTPUT_PATH = "../data/processed/otm_poi_weights.csv"
GROUP_STATS_OUTPUT_PATH = "../data/processed/otm_poi_weight_group_stats.csv"

SHRINKAGE_K = 5
MIN_REL_GROUP = 5

poi_df = pd.read_csv(INPUT_PATH)
print("Input shape:", poi_df.shape)
poi_df[["poi_id", "display_name_en", "category_clean", "query_area", "wiki_has_page", "wiki_pageviews_total"]].head(10)


Input shape: (300, 27)


,poi_id,display_name_en,category_clean,query_area,wiki_has_page,wiki_pageviews_total
0,otm_N7294561685,Chora Mosque / Kariye Museum,museum,Balat / Fener,1,20025.0
1,otm_R1555271,Hagia Sophia,museum,Hagia Sophia / Basilica Cistern,1,1300243.0
2,otm_N415157636,Serpent Column,historic,Sultanahmet Core,1,38432.0
3,otm_R1564032,Süleymaniye Mosque,religious,Suleymaniye,1,138681.0
4,otm_N7215645385,The Blue Mosque,religious,Sultanahmet Core,1,50304.0
5,otm_W103953125,Tomb of Sultan Ahmet,religious,Sultanahmet Core,1,50304.0
6,otm_W326372295,Yıldız Palace,museum,Yildiz Palace,1,39930.0
7,otm_N6879165718,15 July coup monument (Istanbul),historic,Beylerbeyi Palace,1,4959.0
8,otm_N3580821812,Adam Mickiewicz Museum,museum,Istiklal / Pera,1,68.0
9,otm_W109980654,Ahi Çelebi Mosque,religious,Eminonu / Spice Bazaar,1,839.0


## Prepare wiki-based raw signals

In [3]:
work_df = poi_df.copy()
work_df["wiki_has_page"] = work_df["wiki_has_page"].astype(str)
work_df["wiki_pageviews_total"] = pd.to_numeric(work_df["wiki_pageviews_total"], errors="coerce")
work_df["wiki_raw_score"] = np.where(
    work_df["wiki_has_page"] == "1",
    np.log1p(work_df["wiki_pageviews_total"].fillna(0)),
    np.nan,
)

wiki_df = work_df.loc[work_df["wiki_has_page"] == "1"].copy()
print("Wiki-covered POIs:", len(wiki_df))
wiki_df[["display_name_en", "wiki_pageviews_total", "wiki_raw_score"]].head(10)


Wiki-covered POIs: 233


,display_name_en,wiki_pageviews_total,wiki_raw_score
0,Chora Mosque / Kariye Museum,20025.0,9.904787
1,Hagia Sophia,1300243.0,14.078062
2,Serpent Column,38432.0,10.556672
3,Süleymaniye Mosque,138681.0,11.839939
4,The Blue Mosque,50304.0,10.825860
5,Tomb of Sultan Ahmet,50304.0,10.825860
6,Yıldız Palace,39930.0,10.594908
7,15 July coup monument (Istanbul),4959.0,8.509161
8,Adam Mickiewicz Museum,68.0,4.234107
9,Ahi Çelebi Mosque,839.0,6.733402


## Build group statistics from wiki-covered POIs

In [4]:
global_median = float(wiki_df["wiki_raw_score"].median())

category_stats = (
    wiki_df.groupby("category_clean", dropna=False)["wiki_raw_score"]
    .agg(group_median="median", group_count="count")
    .reset_index()
)

category_area_stats = (
    wiki_df.groupby(["category_clean", "query_area"], dropna=False)["wiki_raw_score"]
    .agg(group_median="median", group_count="count")
    .reset_index()
)

category_lookup = {
    row["category_clean"]: (float(row["group_median"]), int(row["group_count"]))
    for _, row in category_stats.iterrows()
}

category_area_lookup = {
    (row["category_clean"], row["query_area"]): (float(row["group_median"]), int(row["group_count"]))
    for _, row in category_area_stats.iterrows()
}

group_stats_df = pd.concat(
    [
        category_stats.assign(group_type="category", query_area=""),
        category_area_stats.assign(group_type="category_area"),
    ],
    ignore_index=True,
)

print("Global wiki raw-score median:", round(global_median, 4))
category_stats.sort_values("group_count", ascending=False).head(10)


Global wiki raw-score median: 7.8793


,category_clean,group_median,group_count
3,religious,7.312185,94
1,historic,8.740657,69
2,museum,7.578353,44
0,attraction,8.315196,26


## Fallback helpers

In [5]:
def compute_lambda(n_group, k=SHRINKAGE_K):
    return float(n_group) / float(n_group + k)


def get_nonreligious_group_prior(category, area):
    key = (category, area)
    if key in category_area_lookup:
        median_score, count = category_area_lookup[key]
        return median_score, count, "category_area"
    if category in category_lookup:
        median_score, count = category_lookup[category]
        return median_score, count, "category"
    return global_median, 0, "global"


def get_religious_group_prior(category, area):
    if category in category_lookup:
        median_score, count = category_lookup[category]
        return median_score, count, "category"

    key = (category, area)
    if key in category_area_lookup:
        median_score, count = category_area_lookup[key]
        if count >= MIN_REL_GROUP:
            return median_score, count, "category_area_cautious"

    return global_median, 0, "global"


def compute_raw_score(row):
    if row["wiki_has_page"] == "1":
        return pd.Series({
            "raw_attractiveness_score": float(row["wiki_raw_score"]),
            "fallback_group_median": np.nan,
            "fallback_group_size": np.nan,
            "lambda_p": 1.0,
            "poi_weight_source": "wiki_direct",
            "poi_weight_confidence": "high",
        })

    category = row["category_clean"]
    area = row["query_area"]

    if category == "religious":
        group_median, group_size, source = get_religious_group_prior(category, area)
        confidence = "low" if source in {"category", "global"} else "medium_low"
    else:
        group_median, group_size, source = get_nonreligious_group_prior(category, area)
        confidence = {
            "category_area": "medium",
            "category": "medium_low",
            "global": "low",
        }[source]

    lambda_p = compute_lambda(group_size)
    raw_score = lambda_p * group_median

    return pd.Series({
        "raw_attractiveness_score": float(raw_score),
        "fallback_group_median": float(group_median),
        "fallback_group_size": int(group_size),
        "lambda_p": float(lambda_p),
        "poi_weight_source": f"fallback_{source}",
        "poi_weight_confidence": confidence,
    })


## Compute POI weights

In [6]:
weight_components = work_df.apply(compute_raw_score, axis=1)
weights_df = pd.concat([work_df, weight_components], axis=1)

total_raw_score = weights_df["raw_attractiveness_score"].sum()
weights_df["poi_weight"] = weights_df["raw_attractiveness_score"] / total_raw_score

print("Total raw attractiveness score:", round(float(total_raw_score), 4))
print("POI weight sum:", round(float(weights_df["poi_weight"].sum()), 6))

weights_df[[
    "display_name_en",
    "category_clean",
    "wiki_has_page",
    "wiki_pageviews_total",
    "raw_attractiveness_score",
    "poi_weight",
    "poi_weight_source",
    "poi_weight_confidence",
]].sort_values("poi_weight", ascending=False).head(20)


Total raw attractiveness score: 1949.5667
POI weight sum: 1.0


,display_name_en,category_clean,wiki_has_page,wiki_pageviews_total,raw_attractiveness_score,poi_weight,poi_weight_source,poi_weight_confidence
241,Istanbul Radio House,attraction,1,1617259.0,14.296244,0.007333,wiki_direct,high
66,"Edirnekapı, Istanbul",historic,1,1617259.0,14.296244,0.007333,wiki_direct,high
99,Istanbul City Wall,historic,1,1617259.0,14.296244,0.007333,wiki_direct,high
1,Hagia Sophia,museum,1,1300243.0,14.078062,0.007221,wiki_direct,high
40,Byzantium,historic,1,296282.0,12.599070,0.006462,wiki_direct,high
184,Topkapı Palace,museum,1,268748.0,12.501533,0.006412,wiki_direct,high
116,Mausoleum of Mahmud II,historic,1,248567.0,12.423472,0.006372,wiki_direct,high
65,Ecumenical Patriarchate of Constantinople,religious,1,217500.0,12.289959,0.006304,wiki_direct,high
189,Walls of Constantinople,historic,1,209372.0,12.251873,0.006284,wiki_direct,high
78,Galata Tower,attraction,1,154494.0,11.947917,0.006128,wiki_direct,high


## Diagnostics

In [7]:
print("Weight source counts:")
print(weights_df["poi_weight_source"].value_counts())

print("\nConfidence counts:")
print(weights_df["poi_weight_confidence"].value_counts())

print("\nTop no-wiki POIs by weight:")
weights_df.loc[weights_df["wiki_has_page"] == "0", [
    "display_name_en",
    "category_clean",
    "query_area",
    "raw_attractiveness_score",
    "poi_weight",
    "poi_weight_source",
    "fallback_group_size",
    "lambda_p",
]].sort_values("poi_weight", ascending=False).head(20)


Weight source counts:
poi_weight_source
wiki_direct               233
fallback_category          43
fallback_category_area     24
Name: count, dtype: int64

Confidence counts:
poi_weight_confidence
high          233
low            42
medium         24
medium_low      1
Name: count, dtype: int64

Top no-wiki POIs by weight:


,display_name_en,category_clean,query_area,raw_attractiveness_score,poi_weight,poi_weight_source,fallback_group_size,lambda_p
12,All Saints Moda English Church,religious,Kadikoy Historic Center,6.942883,0.003561,fallback_category,94.0,0.949495
20,Aya Pandeleimon Rum Ortodoks Kilisesi,religious,Beylerbeyi Palace,6.942883,0.003561,fallback_category,94.0,0.949495
253,Müderris Abdülbaki Efendi Camii,religious,Uskudar Waterfront,6.942883,0.003561,fallback_category,94.0,0.949495
206,Ahmediye Cami,religious,Fatih / Valens Aqueduct,6.942883,0.003561,fallback_category,94.0,0.949495
209,Alman Protestan Kilisesi,religious,Istiklal / Pera,6.942883,0.003561,fallback_category,94.0,0.949495
212,Aya Nikola Kilisesi,religious,Galata Tower,6.942883,0.003561,fallback_category,94.0,0.949495
214,Ayvansaray Panayia Balino Church,religious,Balat / Fener,6.942883,0.003561,fallback_category,94.0,0.949495
221,Burhaniye Camii,religious,Beylerbeyi Palace,6.942883,0.003561,fallback_category,94.0,0.949495
278,Surp Yeyğa Kilisesi,religious,Eyup / Pierre Loti,6.942883,0.003561,fallback_category,94.0,0.949495
228,Dragoman Yunus Beg Mosque,religious,Balat / Fener,6.942883,0.003561,fallback_category,94.0,0.949495


## Save outputs

In [8]:
weights_df.to_csv(OUTPUT_PATH, index=False)
group_stats_df.to_csv(GROUP_STATS_OUTPUT_PATH, index=False)

print("Saved POI weights:", OUTPUT_PATH, weights_df.shape)
print("Saved group stats:", GROUP_STATS_OUTPUT_PATH, group_stats_df.shape)


Saved POI weights: ../data/processed/otm_poi_weights.csv (300, 35)
Saved group stats: ../data/processed/otm_poi_weight_group_stats.csv (66, 5)
